In [35]:
import boto3
import pandas as pd
import os
from datetime import datetime
from dotenv import load_dotenv

# Load environment variables from .env file (if it exists)
load_dotenv()

# Initialize DynamoDB client
aws_region = os.getenv('AWS_REGION', 'eu-north-1')
endpoint_url = os.getenv('AWS_ENDPOINT_URL')
DYNAMODB_TABLE = os.getenv('DYNAMODB_TABLE', 'apollolytics_dialogues')

if endpoint_url:
    dynamodb = boto3.resource('dynamodb', region_name=aws_region, endpoint_url=endpoint_url)
else:
    dynamodb = boto3.resource('dynamodb', region_name=aws_region)

# Get the table
table = dynamodb.Table(DYNAMODB_TABLE)

# Scan the table to get all items
response = table.scan()
items = response['Items']

# Handle pagination if there are more results
while 'LastEvaluatedKey' in response:
    response = table.scan(ExclusiveStartKey=response['LastEvaluatedKey'])
    items.extend(response['Items'])

# Convert to DataFrame
df = pd.DataFrame(items)

# Convert timestamp to datetime for better analysis
df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce')
df['datetime'] = pd.to_datetime(df['timestamp'], unit='s')

# Sort by session_id and timestamp
df = df.sort_values(['session_id', 'timestamp'])

# Display basic info
print(f"Total records: {len(df)}")
print(f"Unique sessions: {df['session_id'].nunique()}")
print(f"\nColumns in the dataset:")
print(df.columns.tolist())

# Display first few rows
df.head()

Total records: 963
Unique sessions: 168

Columns in the dataset:
['event_type', 'created_at', 'session_id', 'timestamp', 'dialogue_mode', 'origin_url', 'article', 'message_content', 'role', 'message_id', 'transcript', 'reason', 'prolific_id', 'propaganda_result', 'content', 'timing_info', 'datetime']


,event_type,created_at,session_id,timestamp,dialogue_mode,origin_url,article,message_content,role,message_id,transcript,reason,prolific_id,propaganda_result,content,timing_info,datetime
461,session_init,2025-05-20T16:34:41.017249,012e16f7-2d9a-416d-8bf9-9b8c1605bf4e,1.747759e+09,critical,http://localhost:3000/dialogue/positive,asd,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-05-20 16:34:41
462,propaganda_analysis,2025-05-20T16:34:42.095693,012e16f7-2d9a-416d-8bf9-9b8c1605bf4e,1.747759e+09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{'type': 'contextualization', 'data': {}, 'use...",NaN,NaN,2025-05-20 16:34:42
463,NaN,2025-05-20T16:35:16.867553,012e16f7-2d9a-416d-8bf9-9b8c1605bf4e,1.747759e+09,NaN,NaN,NaN,NaN,assistant,assistant_608c7248-7681-4072-aa0d-2af926da4e2c,NaN,NaN,NaN,NaN,What are your thoughts on the importance of sk...,"{'model_generation_time': 34.72401285171509, '...",2025-05-20 16:35:16
464,NaN,2025-05-20T16:35:36.156042,012e16f7-2d9a-416d-8bf9-9b8c1605bf4e,1.747759e+09,NaN,NaN,NaN,NaN,user,user_5825deb5-f23e-40fd-9a2b-78e62865bf6c,NaN,NaN,NaN,NaN,I think this conversation has stalled.,"{'thinking_time': 4.693, 'recording_duration':...",2025-05-20 16:35:36
252,session_init,2025-05-26T15:29:07.245837,03b6ca31-3a06-4e39-a4ab-701c3bc53dc7,1.748273e+09,positive,https://apollolytics-dialogue-kxqo4hr1x-kilian...,From welfare to Waffen: Germany’s militarism i...,NaN,NaN,NaN,NaN,NaN,asdfg,NaN,NaN,NaN,2025-05-26 15:29:07


In [32]:
#filter by date newest to oldest
df = df.sort_values(by='timestamp', ascending=False)
# filter for dates 2025-07-14
df = df[df['datetime'] > pd.Timestamp('2025-07-15')]
recent_inits = df[df['event_type'] == 'session_init'].sort_values('datetime', ascending=False)
# drop columns where all values are nan
df = df.dropna(axis=1, how='all')


In [36]:
df[df.prolific_id == '5c921106b788a2000170709f']

,event_type,created_at,session_id,timestamp,dialogue_mode,origin_url,article,message_content,role,message_id,transcript,reason,prolific_id,propaganda_result,content,timing_info,datetime
677,session_init,2025-07-15T13:31:28.015611,12f7327e-a8ee-4d69-b49a-fbedc65e0970,1.752586e+09,critical,https://apollolytics-dialogue.vercel.app/dialo...,Trump admin asking federal agencies to cancel ...,NaN,NaN,NaN,NaN,NaN,5c921106b788a2000170709f,NaN,NaN,NaN,2025-07-15 13:31:28


In [ ]:
df[df.session_id == '12f7327e-a8ee-4d69-b49a-fbedc65e0970']

,event_type,created_at,session_id,timestamp,dialogue_mode,origin_url,article,role,message_id,reason,prolific_id,propaganda_result,content,timing_info,datetime
406,message,2025-07-15T12:56:59.620524,9e767dac-05ab-4eb8-93c6-5c253fc4246e,1.752584e+09,NaN,NaN,NaN,assistant,assistant_cbbf48ff-4ddf-4be0-8ee0-e0c1d0825b70,NaN,NaN,NaN,You mentioned you think Harvard is racist. Tha...,"{'model_generation_time': 13.216657876968384, ...",2025-07-15 12:56:59
405,message,2025-07-15T12:56:41.821855,9e767dac-05ab-4eb8-93c6-5c253fc4246e,1.752584e+09,NaN,NaN,NaN,user,user_af3f5562-dfcf-4708-af3b-fde2fe47512d,NaN,NaN,NaN,"Yeah, I think that Harvard is quite racist.","{'thinking_time': 8.805, 'recording_duration':...",2025-07-15 12:56:41
404,message,2025-07-15T12:56:22.306116,9e767dac-05ab-4eb8-93c6-5c253fc4246e,1.752584e+09,NaN,NaN,NaN,assistant,assistant_5b218257-9691-45fd-83ee-bb46bb9c39ea,NaN,NaN,NaN,It's interesting that you see the efforts as g...,"{'model_generation_time': 8.292434453964233, '...",2025-07-15 12:56:22
403,message,2025-07-15T12:56:12.027336,9e767dac-05ab-4eb8-93c6-5c253fc4246e,1.752584e+09,NaN,NaN,NaN,user,user_d6dadfaa-acd5-49a7-bcc8-97f2f6e09ac9,NaN,NaN,NaN,I think the efforts are good.,"{'thinking_time': 0.973, 'recording_duration':...",2025-07-15 12:56:12
402,message,2025-07-15T12:56:02.020207,9e767dac-05ab-4eb8-93c6-5c253fc4246e,1.752584e+09,NaN,NaN,NaN,assistant,assistant_18935b2a-63ea-4131-adde-75371514796d,NaN,NaN,NaN,What do you think about the Trump administrati...,"{'model_generation_time': 7.463375091552734, '...",2025-07-15 12:56:02
401,propaganda_analysis,2025-07-15T12:55:54.546594,9e767dac-05ab-4eb8-93c6-5c253fc4246e,1.752584e+09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{'type': 'contextualization', 'data': {'Poor J...",NaN,NaN,2025-07-15 12:55:55
400,session_init,2025-07-15T12:55:54.472981,9e767dac-05ab-4eb8-93c6-5c253fc4246e,1.752584e+09,critical,https://apollolytics-dialogue.vercel.app/dialo...,Trump admin asking federal agencies to cancel ...,NaN,NaN,NaN,kdot123,NaN,NaN,NaN,2025-07-15 12:55:54


In [8]:
#!/usr/bin/env python3
"""
Debug script to query DynamoDB and show all events for a specific session.
"""

import boto3
import os
import json
from datetime import datetime

# AWS configuration
aws_region = os.environ.get('AWS_REGION', 'eu-north-1')
DYNAMODB_TABLE = os.environ.get('DYNAMODB_TABLE', 'apollolytics_dialogues')

# Initialize DynamoDB
dynamodb = boto3.resource('dynamodb', region_name=aws_region)
table = dynamodb.Table(DYNAMODB_TABLE)

def query_session_data(session_id: str):
    """Query all data for a specific session."""
    try:
        response = table.query(
            KeyConditionExpression=boto3.dynamodb.conditions.Key('session_id').eq(session_id)
        )
        
        items = response.get('Items', [])
        print(f"\n=== Session Data for {session_id} ===")
        print(f"Total items found: {len(items)}")
        
        for i, item in enumerate(items):
            print(f"\n--- Item {i+1} ---")
            print(f"Session ID: {item.get('session_id')}")
            print(f"Timestamp: {item.get('timestamp')}")
            print(f"Event Type: {item.get('event_type', 'MISSING!')}")
            print(f"Created At: {item.get('created_at')}")
            
            # Show specific fields based on event type
            event_type = item.get('event_type')
            if event_type == 'session_init':
                print(f"Dialogue Mode: {item.get('dialogue_mode')}")
                print(f"Origin URL: {item.get('origin_url')}")
                print(f"Prolific ID: {item.get('prolific_id')}")
                print(f"Article Length: {len(item.get('article', ''))} chars")
            elif event_type == 'propaganda_analysis':
                print(f"Propaganda Result Keys: {list(item.get('propaganda_result', {}).keys())}")
            elif event_type == 'session_end':
                print(f"Reason: {item.get('reason')}")
            else:
                # Message event (no event_type field)
                print(f"Role: {item.get('role')}")
                print(f"Message ID: {item.get('message_id')}")
                print(f"Content: {item.get('content', '')}...")
                print(f"Timing Info: {item.get('timing_info', {})}")
            
            print("-" * 50)
            
        return items
        
    except Exception as e:
        print(f"Error querying session data: {e}")
        return []

def list_recent_sessions(limit=10):
    """List recent sessions."""
    try:
        response = table.scan(
            ProjectionExpression="session_id, #ts, event_type, created_at",
            ExpressionAttributeNames={"#ts": "timestamp"},
            Limit=limit
        )
        
        items = response.get('Items', [])
        print(f"\n=== Recent Sessions (showing up to {limit}) ===")
        
        # Group by session_id and show event types
        sessions = {}
        for item in items:
            session_id = item.get('session_id')
            event_type = item.get('event_type', 'message')
            created_at = item.get('created_at')
            
            if session_id not in sessions:
                sessions[session_id] = {'events': [], 'created_at': created_at}
            sessions[session_id]['events'].append(event_type)
        
        for session_id, data in sessions.items():
            events = data['events']
            created_at = data['created_at']
            print(f"Session: {session_id}")
            print(f"  Created: {created_at}")
            print(f"  Events: {events}")
            print()
            
    except Exception as e:
        print(f"Error listing sessions: {e}")

if __name__ == "__main__":
    # List recent sessions first
    list_recent_sessions(5)
    
    # Query specific session (use the session ID from your logs)
    session_id = "8a52bbf1-2381-44e4-b682-9407be72896c"  # From your logs
    query_session_data(session_id) 


=== Recent Sessions (showing up to 5) ===
Session: 24399df5-600c-4db8-a3e6-a051904ae466
  Created: 2025-05-07T10:14:55.153239
  Events: ['session_init', 'message', 'message', 'message', 'session_end']


=== Session Data for 8a52bbf1-2381-44e4-b682-9407be72896c ===
Total items found: 3

--- Item 1 ---
Session ID: 8a52bbf1-2381-44e4-b682-9407be72896c
Timestamp: 1752580559020550
Event Type: session_init
Created At: 2025-07-15T11:55:59.020553
Dialogue Mode: critical
Origin URL: https://apollolytics-dialogue.vercel.app/dialogue/positive1
Prolific ID: KDOT
Article Length: 5652 chars
--------------------------------------------------

--- Item 2 ---
Session ID: 8a52bbf1-2381-44e4-b682-9407be72896c
Timestamp: 1752580559035008
Event Type: propaganda_analysis
Created At: 2025-07-15T11:55:59.035195
Propaganda Result Keys: ['type', 'data', 'user_id', 'status']
--------------------------------------------------

--- Item 3 ---
Session ID: 8a52bbf1-2381-44e4-b682-9407be72896c
Timestamp: 17525805658

In [ ]:
# Debug script to check what's actually in the database
import boto3
import pandas as pd
import os
from datetime import datetime

# Load environment variables
aws_region = os.getenv('AWS_REGION', 'eu-north-1')
endpoint_url = os.getenv('AWS_ENDPOINT_URL')
DYNAMODB_TABLE = os.getenv('DYNAMODB_TABLE', 'apollolytics_dialogues')

if endpoint_url:
    dynamodb = boto3.resource('dynamodb', region_name=aws_region, endpoint_url=endpoint_url)
else:
    dynamodb = boto3.resource('dynamodb', region_name=aws_region)

table = dynamodb.Table(DYNAMODB_TABLE)

# Get the most recent session_init events
response = table.scan(
    FilterExpression=boto3.dynamodb.conditions.Attr('event_type').eq('session_init'),
    Limit=10
)

items = response['Items']

# Handle pagination
while 'LastEvaluatedKey' in response and len(items) < 10:
    response = table.scan(
        FilterExpression=boto3.dynamodb.conditions.Attr('event_type').eq('session_init'),
        ExclusiveStartKey=response['LastEvaluatedKey'],
        Limit=10-len(items)
    )
    items.extend(response['Items'])

# Convert to DataFrame
df_debug = pd.DataFrame(items)

if len(df_debug) > 0:
    print("Most recent session_init events:")
    print(df_debug[['session_id', 'prolific_id', 'dialogue_mode', 'origin_url', 'created_at']].to_string())
    
    print(f"\nData types:")
    print(df_debug.dtypes)
    
    print(f"\nUnique prolific_ids:")
    print(df_debug['prolific_id'].unique())
    
    print(f"\nUnique dialogue_modes:")
    print(df_debug['dialogue_mode'].unique())
    
    print(f"\nUnique origin_urls:")
    print(df_debug['origin_url'].unique())
else:
    print("No session_init events found in database")

Most recent session_init events:
                             session_id prolific_id dialogue_mode                                                  origin_url                  created_at
0  24399df5-600c-4db8-a3e6-a051904ae466         NaN      critical  https://apollolytics-dialogue.vercel.app/dialogue/positive  2025-05-07T10:14:55.153239
1  21aed941-b5ba-4a94-943e-557478325fb4         NaN      critical                     http://localhost:3000/dialogue/positive  2025-05-14T14:48:16.686450
2  54aab53e-bedb-4a84-aa3b-14557c90d5e2         XXX      critical                     http://localhost:3000/dialogue/positive  2025-05-26T09:44:39.566374
3  083ef2d2-f651-42af-aaf7-24ab6b26dfa1         NaN      critical                     http://localhost:3000/dialogue/positive  2025-05-19T15:08:58.871177
4  8028699e-b7de-4e1e-8b50-cd8c46c9194e         NaN      critical                     http://localhost:3000/dialogue/positive  2025-05-19T17:36:30.020041
5  9477a8c5-0ad2-4ec2-966e-3387986fce42    

In [32]:
# Filter for session_init events before 2025-07-13
cutoff_date = pd.to_datetime("2025-07-13")

df_before_cutoff = df[
    (df['event_type'] == 'session_init') &
    (df['datetime'] < cutoff_date)
]

print(f"Session_init events before 2025-07-13: {len(df_before_cutoff)}")
print(df_before_cutoff[['session_id', 'prolific_id', 'dialogue_mode', 'origin_url', 'datetime']].to_string())
# Example: Filter for experiment sessions with valid prolific IDs before cutoff date
experiment_sessions = df[
    (df['event_type'] == 'session_init') &
    (df['prolific_id'].str.len() == 24) &
    (df['origin_url'].str.contains('positive[123]|negative[123]', na=False)) &
    (df['datetime'] < cutoff_date)
]['session_id'].unique()

df_experiment = df[df['session_id'].isin(experiment_sessions)].copy()
df_experiment = df_experiment.sort_values(['datetime', 'session_id', 'timestamp'], ascending=[False, True, True])

print(f"Filtered experiment dataset before 2025-07-13:")
print(df_experiment[['session_id', 'prolific_id', 'dialogue_mode', 'origin_url', 'datetime']].to_string())

Session_init events before 2025-07-13: 0
Empty DataFrame
Columns: [session_id, prolific_id, dialogue_mode, origin_url, datetime]
Index: []
Filtered experiment dataset before 2025-07-13:
Empty DataFrame
Columns: [session_id, prolific_id, dialogue_mode, origin_url, datetime]
Index: []


In [3]:
# Filter for sessions with valid prolific IDs (24 characters long)
valid_prolific_sessions = df[
    (df['event_type'] == 'session_init') & 
    (df['prolific_id'].str.len() == 24)
]['session_id'].unique()

print(f"Found {len(valid_prolific_sessions)} sessions with valid prolific IDs")

# Filter the main dataframe to only include sessions with valid prolific IDs
df_valid_prolific = df[df['session_id'].isin(valid_prolific_sessions)].copy()

# Sort by session_id and timestamp
df_valid_prolific = df_valid_prolific.sort_values(['session_id', 'timestamp'])

# Display summary
print(f"\nFiltered dataset:")
print(f"Total records: {len(df_valid_prolific)}")
print(f"Unique sessions: {df_valid_prolific['session_id'].nunique()}")

# Show unique prolific IDs
valid_prolific_ids = df_valid_prolific[
    (df_valid_prolific['event_type'] == 'session_init') & 
    (df_valid_prolific['prolific_id'].str.len() == 24)
]['prolific_id'].unique()

print(f"\nValid prolific IDs found:")
for prolific_id in valid_prolific_ids:
    print(f"  - {prolific_id}")

# Display first few rows of filtered data
df_valid_prolific.head(10)

Found 0 sessions with valid prolific IDs

Filtered dataset:
Total records: 0
Unique sessions: 0

Valid prolific IDs found:


,event_type,created_at,session_id,timestamp,dialogue_mode,origin_url,article,message_content,role,message_id,transcript,reason,prolific_id,propaganda_result,content,timing_info,datetime


In [26]:
# Check ALL prolific IDs in the database, regardless of length
all_prolific_ids = df[
    (df['event_type'] == 'session_init') & 
    (df['prolific_id'].notna())
]['prolific_id'].unique()

print(f"All prolific IDs found (any length):")
for prolific_id in all_prolific_ids:
    print(f"  - '{prolific_id}' (length: {len(prolific_id)})")

# Check for any recent sessions from experiment routes
recent_experiment_sessions = df[
    (df['event_type'] == 'session_init') & 
    (df['origin_url'].str.contains('positive[123]|negative[123]', na=False))
]

print(f"\nRecent experiment sessions:")
if len(recent_experiment_sessions) > 0:
    print(recent_experiment_sessions[['session_id', 'prolific_id', 'origin_url', 'datetime']].to_string())
else:
    print("No experiment sessions found")

# Check the most recent sessions overall
recent_sessions = df[
    df['event_type'] == 'session_init'
].sort_values('datetime', ascending=False).head(5)

print(f"\nMost recent 5 sessions:")
print(recent_sessions[['session_id', 'prolific_id', 'dialogue_mode', 'origin_url', 'datetime']].to_string())

All prolific IDs found (any length):
  - 'XXX' (length: 3)
  - 'saf' (length: 3)

Recent experiment sessions:
                               session_id prolific_id                                                   origin_url            datetime
445  56a30a89-698d-4161-83a9-bf8ea76aa7d9         saf  https://apollolytics-dialogue.vercel.app/dialogue/negative3 2025-07-13 16:41:05

Most recent 5 sessions:
                               session_id prolific_id dialogue_mode                                                   origin_url            datetime
312  3924414e-e212-48c8-a4bd-b0f61b2532ff         XXX      critical   https://apollolytics-dialogue.vercel.app/dialogue/positive 2025-07-13 19:56:14
445  56a30a89-698d-4161-83a9-bf8ea76aa7d9         saf    supportive  https://apollolytics-dialogue.vercel.app/dialogue/negative3 2025-07-13 16:41:05


In [7]:
df.origin_url.unique()

array(['http://localhost:3000/dialogue/positive', nan,
       'https://apollolytics-dialogue-kxqo4hr1x-kilian-sprenkamps-projects.vercel.app/dialogue/positive1',
       'https://apollolytics-dialogue.vercel.app/dialogue/negative',
       'https://apollolytics-dialogue.vercel.app/dialogue/positive',
       'https://apollolytics-dialogue-kxqo4hr1x-kilian-sprenkamps-projects.vercel.app/dialogue/negative1',
       'https://apollolytics-dialogue.vercel.app/dialogue/negative3',
       'https://apollolytics-dialogue.vercel.app/dialogue/positive1',
       'https://apollolytics-dialogue-kxqo4hr1x-kilian-sprenkamps-projects.vercel.app/dialogue/positive3',
       'http://localhost:3000/dialogue/positive2'], dtype=object)